# 🦥 SFT — MDR Narrative Intelligence (Qwen3.5-4B)

Same pipeline as the Qwen3-0.6B notebook: same schema, same data, same parser.
Four things had to change for Qwen3.5 and each is commented where it appears —
the install pins, the loader class, the LoRA target list, and the batch size.

In [ ]:
!nvidia-smi

## 🦥 Install

In [ ]:
%%capture
# NOTE: %%capture hides pip failures too. If a later cell says a package is missing,
# re-run this with the %%capture line commented out.
import os, sys, importlib.util

!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    !uv pip install -qqq "torch==2.8.0" "triton>=3.3.0" numpy pillow torchvision \
        xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

# bitsandbytes  -> no 4-bit without it, and 4B does not fit a T4 in 16-bit
# cut_cross_entropy -> Qwen3.5's vocab is 248,320, so fp32 logits cost 0.95 MiB
#                      PER TOKEN. Without CCE a 2,900-token sequence needs 2.7 GiB.
!uv pip install --no-deps bitsandbytes cut_cross_entropy
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0

# causal_conv1d builds only against torch 2.8. On a newer torch it compiles from
# source and takes ~10 min — it does not fail, it just sits there.
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

# torchao <0.15 on purpose. It declares no torch pin, so ">=0.16" resolves to a build
# needing torch>=2.11 and then `import transformers` dies with
#     AttributeError: module 'torchao' has no attribute 'float8'
!uv pip install --no-deps "torchao==0.14.1"

import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"

!pip install -qqq sentencepiece protobuf "datasets>=4.0.0" "huggingface_hub>=0.34.0" hf_transfer

In [ ]:
# Pins must be in THIS interpreter. Metadata is not enough: transformers 5.x pulls
# in the torchao quantizer at import time, so a torchao/torch mismatch only shows up
# when you actually import it.
import sys, importlib, importlib.metadata as md

print("python:", sys.version.split()[0], "|", sys.executable)

bad = []
for pkg, want in (("transformers", "5.2.0"), ("trl", "0.22.2"), ("torch", "2.8.0"),
                  ("torchao", "0.14.1"), ("unsloth", None), ("datasets", None)):
    try:
        got = md.version(pkg)
    except Exception:
        got = None
    ok = got is not None and (want is None or got == want)
    print(f"  {pkg:<14} {got or 'MISSING':<12}{'' if ok else '  <-- want ' + str(want)}")
    if not ok:
        bad.append(pkg)
assert not bad, f"wrong/missing: {bad}. Install with `{sys.executable} -m pip`, not bare pip."

for mod in ("torch", "transformers"):
    importlib.import_module(mod)          # raises here, not three cells later
print("  imports        OK")

# Optional speed kernels. transformers wraps the Gated DeltaNet ops with
# use_kernel_func_from_hub_with_fallback(..., "fla"), which falls back to a pure
# torch path. Missing = slower, not broken.
for mod in ("fla", "causal_conv1d"):
    try:
        importlib.import_module(mod)
        print(f"  {mod:<14} fast kernel active")
    except Exception as e:
        print(f"  {mod:<14} unavailable ({type(e).__name__}) — torch fallback, slower")
        print(f"  {'':<14} fix: FLA_TILELANG=0, or `uv pip install -U 'tilelang>=0.1.9'`")

In [ ]:
import unsloth
from unsloth import is_bfloat16_supported   # FastLanguageModel may not exist on a
import os
import json, re, random
import numpy as np
import transformers
   
os.environ["TOKENIZERS_PARALLELISM"] = "true"
# 4-bit weights, a 248k-vocab logits tensor and checkpointed activations are three
# very differently sized allocations, which fragments the caching allocator. The T4
# OOM reported 2.30 GiB "reserved but unallocated" — that is fragmentation.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# dill (<=0.4.1) overrides Pickler._batch_setitems with the pre-3.14 two-arg
# signature, so `datasets` dies fingerprinting a dict on Python 3.14. No-op below.
try:
    import inspect, dill
    _orig = dill._dill.Pickler._batch_setitems
    if len(inspect.signature(_orig).parameters) == 2:
        dill._dill.Pickler._batch_setitems = lambda self, items, obj=None: _orig(self, items)
except Exception as e:
    print("dill shim skipped:", type(e).__name__)

import torch
torch._dynamo.config.disable = True          # dynamo is unstable on Kaggle images
for v in ("TORCHDYNAMO_DISABLE", "TORCH_COMPILE_DISABLE", "UNSLOTH_DISABLE_COMPILE"):
    os.environ[v] = "1"

                                         # VLM build — the loader cell picks it

seed = 3407
transformers.set_seed(seed)
torch.manual_seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB)  bf16={is_bfloat16_supported()}")

## 🦥 The schema

**This block is duplicated verbatim in notebook 2.** Both notebooks must run standalone
on Kaggle with no shared import, so the schema, the prompt builder and the parser are
copied rather than imported.

`SCHEMA_VERSION` is the guard against them drifting apart. This notebook writes it into
the Hub repo alongside the model, and notebook 2 asserts its own copy matches. Without
that check, an edit to one copy would leave GRPO scoring every well-formed rollout zero,
silently.

In [ ]:
SCHEMA_VERSION = "mni-v1"

SYSTEM_PROMPT = """You are a medical-device adverse-event analyst. You read an FDA MAUDE report and produce one structured record.

Reply in EXACTLY this format, in this order, one field per line:

<think>
Two or three sentences: what failed, what happened to the patient, what the manufacturer did about it.
</think>
FAILURE_MODE: <short phrase>
MECHANISM: <cause> -> <effect>
PATIENT_HARM: <short phrase, or "none">
INTERVENTION: <short phrase, or "none stated">
OUTCOME: <short phrase, or "not stated">
DEVICE_ROLE: <defect established | present at harm site | no device problem identified | insufficient information>
IMDRF_A: <code> <term> | <code> <term>
IMDRF_E: <code> <term> | <code> <term>
EVIDENCE: <code> <- "<verbatim quote from the report>"
MFR_INVESTIGATION: <device examined | records review only | none stated | not applicable>
DEVICE_EXAMINED: <true | false | unknown>
ADDRESSES_HARM: <true | false | not applicable>
FLAG: <short phrase, or "none">

Rules:
- Every EVIDENCE quote must be copied verbatim from the report. Never paraphrase a quote.
- Give one EVIDENCE line per IMDRF code you state.
- Use only the IMDRF codes and terms supplied to you.
- Keep every field on one line."""

FIELDS = ["FAILURE_MODE", "MECHANISM", "PATIENT_HARM", "INTERVENTION", "OUTCOME",
          "DEVICE_ROLE", "IMDRF_A", "IMDRF_E", "EVIDENCE", "MFR_INVESTIGATION",
          "DEVICE_EXAMINED", "ADDRESSES_HARM", "FLAG"]

CODE_RE = re.compile(r"\b(\d{2,5})\b")


def parse_record(text):
    """Parse a completion into fields. None if the schema is not satisfied.

    EVIDENCE accumulates; every other field keeps its first occurrence. The model emits
    one EVIDENCE line per code, so treating it like a single-valued field would keep
    only the first quote — which later lets the grounding reward score one quote and
    ignore the rest.
    """
    body = text.split("</think>")[-1] if "</think>" in text else text
    out, evidence = {}, []
    for line in body.splitlines():
        line = line.strip()
        if not line or ":" not in line:
            if evidence and line.startswith(("-", "*")):
                evidence.append(line)
            continue
        k, _, v = line.partition(":")
        k = k.strip().upper()
        if k == "EVIDENCE":
            evidence.append(v.strip())
        elif k in FIELDS:
            out.setdefault(k, v.strip())
    if evidence:
        out["EVIDENCE"] = "\n".join(evidence)
    if not all(f in out for f in FIELDS):
        return None
    out["_A"] = set(CODE_RE.findall(out["IMDRF_A"]))
    out["_E"] = set(CODE_RE.findall(out["IMDRF_E"]))
    out["_QUOTES"] = re.findall(r'"([^"]{8,})"', out["EVIDENCE"])
    return out


print("schema", SCHEMA_VERSION, "|", len(FIELDS), "fields")

## 🦥 Hardware profile

One notebook, two jobs: a **smoke test** on Kaggle/Colab to prove the pipeline runs, and
a **full run** on the Lambda A100 that produces the model you actually ship. Rather than
edit constants between them, the profile is derived from the card.

The differences are not cosmetic:

| | T4 / small (smoke) | A100 / large (full) |
|---|---|---|
| precision | fp16 — Turing has no bf16 | bf16 |
| model | Qwen3.5-4B | Qwen3-4B |
| 4-bit | not needed at 0.6B | not needed at 4B in bf16 |
| epochs | 1, capped by `SMOKE_STEPS` | 2, full |
| batch | 2 x 8 | 8 x 2 |

Set `FORCE_PROFILE` to override the detection.

In [ ]:
FORCE_PROFILE = None          # None = auto, or "smoke" / "full"
SMOKE_STEPS   = 200

MODEL_NAME = "unsloth/Qwen3.5-4B"      # same model on both profiles
HF_REPO    = "RP-360/Qwen3.5-4B-mdr-narrative-sft"

vram = bf16 = sm = 0
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16 = is_bfloat16_supported()
    sm   = torch.cuda.get_device_capability()[0]

PROFILE = FORCE_PROFILE or ("full" if (vram >= 38 and bf16) else "smoke")

if PROFILE == "full":                  # A100-40 and up: 9.3 GB of bf16 weights
    LOAD_IN_4BIT = False
    BATCH, ACCUM, EPOCHS, MAX_STEPS = 2, 8, 2, -1
    LORA_R, max_seq_length = 32, 3072
    EVAL_STEPS, SAVE_STEPS = 100, 100
else:                                  # 16 GB T4: 4-bit and a shorter window
    LOAD_IN_4BIT = True
    BATCH, ACCUM, EPOCHS, MAX_STEPS = 1, 16, 2, SMOKE_STEPS
    LORA_R, max_seq_length = 16, 2048
    EVAL_STEPS, SAVE_STEPS = 10, 20

# Window sizing, measured on 1,500 real examples with THIS tokenizer:
#     p50 1056   p90 1549   p95 1857   p99 2624   max 5999
#     2048 keeps 97.0% whole,  2560 -> 98.8%,  3072 -> 99.5%
# SFT trains on the response only, so an example whose PROMPT alone overflows the
# window teaches nothing at all — at 2048 that is 1.6% of the set.
print(f"PROFILE {PROFILE.upper()}  |  {MODEL_NAME}  4bit={LOAD_IN_4BIT}")
print(f"  batch   {BATCH} x {ACCUM} = {BATCH*ACCUM} effective   max_seq {max_seq_length}")
print(f"  steps   {MAX_STEPS if MAX_STEPS > 0 else str(EPOCHS) + ' epochs'}"
      f"   eval/save every {EVAL_STEPS}/{SAVE_STEPS}")
print(f"  push to {HF_REPO}")
if sm and sm < 8:
    print(f"  WARNING sm{sm}x: no tilelang below sm80, so the 24 Gated DeltaNet "
          f"layers take a slow fallback path.")

## 🦥 Load the data

Resolves the Kaggle dataset path first, then a local `data/` folder, so the same
notebook runs in both places unchanged.

In [ ]:
DATA_DIR = None
for base in ("/kaggle/input/mni-sft-grpo", "../data", "data", "."):
    if os.path.exists(os.path.join(base, "sft_train.jsonl")):
        DATA_DIR = base
        break
if DATA_DIR is None:
    raise FileNotFoundError(
        "sft_train.jsonl not found. On Kaggle add the dataset as `mni-sft-grpo`; "
        "locally put the file in ./data/")
print("data:", DATA_DIR)


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


train_rows = load_jsonl(os.path.join(DATA_DIR, "sft_train.jsonl"))
val_rows   = load_jsonl(os.path.join(DATA_DIR, "sft_val.jsonl"))
print(f"train {len(train_rows):,}   val {len(val_rows):,}")
print("\n--- one example -------------------------------------------")
print(train_rows[0]["prompt"][:400], "\n...")
print("\n--- its target -------------------------------------------")
print(train_rows[0]["completion"][:500])

## 🦥 Build the chat-formatted dataset

Two details that matter.

`enable_thinking=True` on the template: our targets carry their own `<think>` block as
literal text, which is what we want the model to learn to emit. Letting the template
also open a thinking turn would nest one inside the other.

Loss is computed on the **response only** — see the `train_on_responses_only` cell
below. Training on the prompt too would spend most of the gradient teaching the model to
reproduce MAUDE narratives, which is not the task.

In [ ]:
from datasets import Dataset

# The 0.6B notebook loaded a whole throwaway model here just to get the chat
# template. At 9.3 GB that is minutes of download and a needless allocation --
# and on a VLM it returns a processor, not a tokenizer. Fetch the tokenizer.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_text(row):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": row["prompt"]},
            {"role": "assistant", "content": row["completion"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=True)


train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds   = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

lens = sorted(len(tokenizer(t["text"], add_special_tokens=False)["input_ids"])
              for t in train_ds.select(range(min(1500, len(train_ds)))))
p = lambda q: lens[min(len(lens) - 1, int(len(lens) * q))]
print(f"tokens/example  p50={p(.5)}  p90={p(.9)}  p95={p(.95)}  p99={p(.99)}  max={lens[-1]}")
print(f"fit within max_seq_length={max_seq_length}: "
      f"{100 * sum(1 for x in lens if x <= max_seq_length) / len(lens):.1f}%")
print("\n--- formatted sample ---")
print(train_ds[0]["text"][:700])

## 🦥 Load the model and attach LoRA

In [ ]:
# Qwen3.5-4B is `Qwen3_5ForConditionalGeneration` — a VLM, even though we only feed
# it text. unsloth routes VLMs through FastVisionModel; older builds also accept
# FastLanguageModel. Try both so the notebook runs either way.
from unsloth import FastVisionModel
try:
    from unsloth import FastLanguageModel
except ImportError:
    FastLanguageModel = None

FastModel, LOADER_NAME, errors = None, None, {}
for cls in (FastLanguageModel, FastVisionModel):
    if cls is None:
        continue
    try:
        model, tokenizer = cls.from_pretrained(
            model_name                 = MODEL_NAME,
            max_seq_length             = max_seq_length,
            load_in_4bit               = LOAD_IN_4BIT,
            use_gradient_checkpointing = "unsloth",
        )
        FastModel, LOADER_NAME = cls, cls.__name__
        break
    except Exception as e:
        errors[cls.__name__] = f"{type(e).__name__}: {str(e)[:120]}"

assert FastModel is not None, errors
print("loaded with:", LOADER_NAME, "|", errors or "no fallback needed")

# FastVisionModel returns a PROCESSOR, whose __call__ is (images, text, videos, ...).
# So `tokenizer(text, ...)` binds the prompt to `images` and the image processor
# tries to base64-decode a chat template. Unwrap once; every later cell then behaves
# like the Qwen3-0.6B notebook.
processor = tokenizer
tokenizer = getattr(processor, "tokenizer", processor)
print("tokenizer  :", type(tokenizer).__name__, "(from", type(processor).__name__ + ")")

probe = tokenizer.apply_chat_template(
    [{"role": "system", "content": "s"}, {"role": "user", "content": "u"}],
    tokenize=False, add_generation_prompt=True, enable_thinking=True)
assert "<|im_start|>" in probe and "<think>" in probe, \
    "chat template lost in the unwrap — use `processor(text=...)` instead"

# 4-bit must actually apply. A T4 run showed 9.29 GB resident (= 16-bit) with
# LOAD_IN_4BIT=True set but bitsandbytes missing, and OOM'd cells later.
if torch.cuda.is_available():
    gb = torch.cuda.memory_allocated() / 1e9
    print(f"weights    : {gb:.2f} GB")
    assert not (LOAD_IN_4BIT and gb > 5.0), (
        f"LOAD_IN_4BIT=True but {gb:.2f} GB resident — that is 16-bit. "
        f"Install bitsandbytes and restart the kernel.")

### 🦥 LoRA targets — this is where Qwen3.5 differs most from Qwen3

In [ ]:
# Qwen3.5-4B interleaves two attention types, and the Qwen3 target list silently
# misses one of them:
#     8 of 32 layers   self_attn.{q,k,v,o}_proj                    full attention
#    24 of 32 layers   linear_attn.{in_proj_qkv,_a,_b,_z,out_proj} Gated DeltaNet
# Adapting only the first 8 trains fine and produces a badly under-adapted model.
# (conv1d / A_log / dt_bias are not nn.Linear, so PEFT cannot wrap them. Expected.)
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "in_proj_qkv", "in_proj_a", "in_proj_b", "in_proj_z", "out_proj",
                "gate_proj", "up_proj", "down_proj"]

kw = dict(r=LORA_R, target_modules=LORA_TARGETS, lora_alpha=LORA_R,
          lora_dropout=0, bias="none",
          use_gradient_checkpointing="unsloth", random_state=seed)
if LOADER_NAME == "FastVisionModel":
    kw.update(finetune_vision_layers=False, finetune_language_layers=True,
              finetune_attention_modules=True, finetune_mlp_modules=True)

model = FastModel.get_peft_model(model, **kw)
model.print_trainable_parameters()

In [ ]:
# Adapters must land on BOTH attention types — this is the check that catches the
# Qwen3 target list being carried over unchanged.
from collections import Counter

kinds = Counter()
for name, _ in model.named_modules():
    if "lora_A" not in name:
        continue
    for key in ("self_attn", "linear_attn", "mlp", "visual"):
        if f".{key}." in name or key in name:
            kinds[key] += 1
            break

print(dict(kinds))
assert kinds["linear_attn"], "no adapter on any linear-attention layer — 24 of 32 " \
                             "layers would train nothing. Check LORA_TARGETS."
assert kinds["self_attn"], "no adapter on the full-attention layers."

## 🦥 SFT config

In [ ]:
from trl import SFTConfig, SFTTrainer

REPORT_TO = "none"        # or "tensorboard" -> event files in sft_outputs/runs/

sft_config = SFTConfig(
    output_dir         = "sft_outputs",
    dataset_text_field = "text",
    # `max_length`, NOT `max_seq_length`. trl 0.22.2 renamed it with no deprecation
    # shim, so the old name is ignored and the 1024 default truncates straight
    # through the assistant target.
    max_length         = max_seq_length,
    packing            = False,          # one report per example

    per_device_train_batch_size = BATCH,
    gradient_accumulation_steps = ACCUM,
    num_train_epochs            = EPOCHS,
    max_steps                   = MAX_STEPS,   # -1 on full = use epochs
    learning_rate               = 2e-4,        # GRPO later uses 5e-6
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "cosine",
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,

    fp16 = not is_bfloat16_supported(),        # T4 is Turing: fp16, never bf16
    bf16 = is_bfloat16_supported(),

    # Eval must not gather logits. The default collects predictions and upcasts to
    # fp32 — [1, 3072, 248320] is 3.0 GB for a SINGLE batch, so training fits and
    # then eval OOMs the card.
    prediction_loss_only       = True,
    per_device_eval_batch_size = 1,
    eval_accumulation_steps    = 1,

    logging_steps    = 10,
    eval_strategy    = "steps",
    eval_steps       = EVAL_STEPS,
    save_steps       = SAVE_STEPS,
    save_total_limit = 2,

    # Keep the BEST checkpoint, not the last. Otherwise a run that overfits in its
    # second epoch ships the overfitted weights.
    #
    # eval_loss is a PROXY: much of it is boilerplate every target shares, so lower
    # loss does not strictly mean better codes. The real gate is the schema
    # pass-rate cell after training. Still better than taking the last step blindly.
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,
    report_to              = REPORT_TO,
    seed                   = seed,
)
print(f"fp16={sft_config.fp16}  bf16={sft_config.bf16}  report_to={REPORT_TO}")

In [ ]:
trainer = SFTTrainer(
    model           = model,
    processing_class = tokenizer,
    args            = sft_config,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
)

### 🦥 Compute loss on the response only

Without this the model spends most of its gradient learning to reproduce MAUDE
narratives — the prompt is far longer than the target. We want loss only on what the
model is meant to generate.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Qwen3 chat template markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Verify the mask: the decoded labels should start at the target, not the narrative.
ex = trainer.train_dataset[0]
labels = [t for t in ex["labels"] if t != -100]
print("supervised span begins:\n", tokenizer.decode(labels)[:300])

### 🦥 Early stopping (in steps) and a JSONL run log

In [ ]:
# ── step-based early stopping + a JSONL run log ──────────────────────────────
import os, sys, time, json, torch
from transformers import TrainerCallback, EarlyStoppingCallback

os.makedirs(sft_config.output_dir, exist_ok=True)
LOG_PATH = os.path.join(sft_config.output_dir, "train.jsonl")


class LogToFile(TrainerCallback):
    """One JSON object per line: `tail -f`, or pd.read_json(path, lines=True).

    Two hooks, not one: HF fires on_log a second time with the eval metrics, and it
    does so BEFORE on_evaluate updates state.best_metric. Logging eval rows from
    on_log would drop `best_so_far` — the only column showing early stopping decide.
    """

    def __init__(self, path=LOG_PATH):
        self.t0, self.fh = time.time(), open(path, "a", buffering=1)
        self.best = None

    def _emit(self, kind, state, payload):
        rec = {"kind": kind, "step": state.global_step,
               "epoch": round(state.epoch or 0, 3),
               "h": round((time.time() - self.t0) / 3600, 3), **payload}
        line = json.dumps(rec, default=float)
        print(line, file=sys.__stderr__, flush=True)   # nohup sees this immediately
        self.fh.write(line + "\n")

    def _peak_gb(self):
        # max_memory_allocated, not memory_allocated: on_log runs after the cache is
        # released, so the live reading is the trough. It showed 1.4/16 GB on a T4
        # right up to the step that OOM'd at 14.3.
        if not torch.cuda.is_available():
            return None
        gb = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        return round(gb, 2)

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and not any(k.startswith("eval_") for k in logs):
            self._emit("train", state, {**logs, "peak_gb": self._peak_gb()})

    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if not metrics:
            return
        # state.best_metric is NOT updated yet — the Trainer calls
        # _determine_best_metric AFTER on_evaluate returns, so reading it here is
        # None on the first evaluation and one behind on every later one. Track the
        # running best ourselves so the log means what it says.
        key = args.metric_for_best_model or "eval_loss"
        key = key if key.startswith("eval_") else "eval_" + key
        val = metrics.get(key)
        better = val is not None and (
            self.best is None or
            (val > self.best if args.greater_is_better else val < self.best))
        if better:
            self.best = val
        self._emit("eval", state, {**metrics, "best_so_far": self.best,
                                   "is_best": better})

    def on_train_end(self, args, state, control, **kw):
        self._emit("end", state, {"best_metric": state.best_metric,
                                  "best_ckpt": state.best_model_checkpoint})
        self.fh.close()


# Patience counts EVALUATIONS. It is step-based only because eval_strategy="steps",
# so the real budget is patience x EVAL_STEPS steps with no better eval_loss.
# num_train_epochs does not enter into it.
EARLY_STOP_PATIENCE  = 5 if PROFILE == "full" else 3
EARLY_STOP_THRESHOLD = 1e-3     # the 0.6B run's last 50 steps gained 0.0031 total

trainer.add_callback(LogToFile())
trainer.add_callback(EarlyStoppingCallback(EARLY_STOP_PATIENCE, EARLY_STOP_THRESHOLD))

# load_best_model_at_end needs a checkpoint at every eval it might pick. HF does
# raise for this, but only after the first eval — a long way into the run.
assert sft_config.save_steps % sft_config.eval_steps == 0, "save_steps must be a multiple of eval_steps"
assert str(sft_config.eval_strategy).lower().endswith("steps"), "patience is only step-based with eval_strategy='steps'"

print(f"log        : {LOG_PATH}")
print(f"early stop : {EARLY_STOP_PATIENCE} evals x {sft_config.eval_steps} steps "
      f"= {EARLY_STOP_PATIENCE * sft_config.eval_steps:,} steps without "
      f"eval_loss improving by > {EARLY_STOP_THRESHOLD}")
print(f"callbacks  : {[type(c).__name__ for c in trainer.callback_handler.callbacks]}")

In [ ]:
import torch
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM before training: {used:.1f} / {total:.1f} GB")

## 🦥 Train

In [ ]:
stats = trainer.train()
print(stats)

# Which checkpoint actually got loaded back, and how it compared.
bs = trainer.state.best_metric
bc = trainer.state.best_model_checkpoint
print(f"\nbest eval_loss   : {bs}")
print(f"best checkpoint  : {bc}")
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
if evals:
    print(f"eval_loss trace  : " +
          "  ".join(f"{h['step']}:{h['eval_loss']:.4f}" for h in evals[-8:]))
    if evals[-1]["eval_loss"] > (bs or 0) * 1.02:
        print("  ⚠ the LAST checkpoint was worse than the best — "
              "load_best_model_at_end just saved you from shipping it.")


## 🦥 Sanity check before pushing

The number that matters is not the loss — it is **schema pass rate**: what fraction of
generations parse into all 13 fields. That is the quantity GRPO needs to be non-zero,
and it is the go/no-go gate for notebook 2.

  * pass rate **> 80%** — GRPO has little headroom left on format; its gains will come
    from code accuracy and grounding
  * pass rate **< 10%** — GRPO will still see no reward variance. Fix the SFT set or
    train longer before going on

In [ ]:
from tqdm import tqdm
FastModel.for_inference(model)

def generate(prompt, max_new_tokens=420):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True,
                                         enable_thinking=True)
    ids = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=0.7,
                         top_p=0.9, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)



N_CHECK = 40
ok = 0
first = None

for r in tqdm(val_rows[:N_CHECK], desc="Checking validation records"):
    gen = generate(r["prompt"])

    if first is None:
        first = gen

    if parse_record(gen) is not None:
        ok += 1

print(f"Valid: {ok}/{N_CHECK} ({ok/N_CHECK:.1%})")

print(f"schema pass rate: {ok}/{N_CHECK} = {100*ok/N_CHECK:.1f}%")
print("\n--- one generation -------------------------------------")
print(first)

## 🦥 Merge, then push to the Hub

`save_pretrained_merged` folds LoRA A into the base weights. Notebook 2 loads the
**merged** model, so when TRL disables the fresh adapter to build its KL reference it
gets this SFT model rather than base Qwen3.

`schema.json` travels with the weights so notebook 2 can assert its copy of the prompt
and parser still matches the one the model was trained on.

In [ ]:
# HF_TOKEN, in order: Kaggle Secrets -> environment -> a .env file.
# The .env branch exists because nothing reads a .env on its own, and python-dotenv
# may not be on the image.
def find_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        if (t := UserSecretsClient().get_secret("HF_TOKEN")):
            return t.strip(), "Kaggle Secrets"
    except Exception:
        pass

    if (t := os.environ.get("HF_TOKEN")):
        return t.strip(), "environment"

    paths = [os.path.join(d, ".env") for d in
             # ../.. matters: this notebook lives in SFT-GRPO/qwen3.5-4b/, so a
             # .env at the repo root is two levels up, not one.
             ([DATA_DIR] if "DATA_DIR" in globals() else [])
             + [".", "..", "../..", "../../..", "data"]]
    if os.path.isdir("/kaggle/input"):        # attached datasets are mounted here
        paths += [f"/kaggle/input/{d}/.env" for d in os.listdir("/kaggle/input")]

    for path in map(os.path.normpath, paths):
        if not os.path.isfile(path):
            continue
        for line in open(path):
            line = line.strip().removeprefix("export ")
            k, _, v = line.partition("=")
            if k.strip() == "HF_TOKEN" and (v := v.strip().strip("'\"")):
                return v, f"the .env at {path}"
    return None, "NOT FOUND. looked in: " + ", ".join(map(os.path.normpath, paths))


HF_TOKEN, src = find_hf_token()
print("HF token:", src)
assert HF_TOKEN, (
    "HF_TOKEN not found. Kaggle: Add-ons -> Secrets -> HF_TOKEN. "
    "Locally: export HF_TOKEN=..., or put it in a .env beside the notebook.")
print(f"  {HF_TOKEN[:3]}...{HF_TOKEN[-3:]} ({len(HF_TOKEN)} chars)")   # fingerprint only

# Export it. Several libraries resolve the token AMBIENTLY rather than taking it as
# an argument — huggingface_hub.get_token() reads HF_TOKEN, and transformers calls
# list_repo_tree() with no token at all, which 401s on a private repo even when
# every call we make passes token= explicitly.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN   # older libraries read this name

In [ ]:
if PROFILE == "smoke":
    print("⚠ SMOKE MODE: pushing a smoke-trained model would leave notebook 2 pointing "
          "at a model that never really trained.\n"
          "  Set PUSH_ANYWAY = True below only if you are deliberately testing the push.")
PUSH_ANYWAY = False
assert PROFILE == "full" or PUSH_ANYWAY, "refusing to push a smoke-mode model"

MERGED_DIR = "sft_merged_16bit"

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("merged →", MERGED_DIR)

with open(os.path.join(MERGED_DIR, "schema.json"), "w") as f:
    json.dump({"schema_version": SCHEMA_VERSION,
               "system_prompt": SYSTEM_PROMPT,
               "fields": FIELDS,
               "base_model": MODEL_NAME,
               "stage": "sft"}, f, indent=2)
print("wrote schema.json")

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO, repo_type="model",
                  commit_message=f"SFT merged 16-bit ({SCHEMA_VERSION}), base {MODEL_NAME}")

info = api.model_info(HF_REPO, files_metadata=True)
print(f"pushed → {HF_REPO}  rev {info.sha[:8]}")
for s in sorted(info.siblings, key=lambda x: -(x.size or 0))[:8]:
    print(f"  {s.rfilename:32s} {(s.size or 0)/2**20:8.1f} MB")

### 🦥 Verify the push round-trips

Loading it straight back is worth the two minutes. A repo that cannot be loaded is
found here, not thirty minutes into notebook 2.

In [ ]:
from huggingface_hub import hf_hub_download

sch = json.load(open(hf_hub_download(HF_REPO, "schema.json", token=HF_TOKEN)))
assert sch["schema_version"] == SCHEMA_VERSION, sch
print("schema.json on the Hub:", sch["schema_version"], "✓")
print("\nNotebook 2 should set:")
print(f'    SFT_REPO = "{HF_REPO}"')